# Notebook 11 — Function calling e integrazione con servizi esterni

**Modulo:** 11 — Costruire con le API
**Tempo stimato:** 3-4 ore

## Cosa costruirai

- Una funzione finta `cerca_url_sospetto(url)` che simula un controllo di
  sicurezza tipo VirusTotal
- Uno schema di function calling reale, cablato con l'API di Anthropic (tool use)
- Il ciclo completo: richiesta → il modello chiede di chiamare una funzione →
  il tuo codice la esegue → il risultato torna al modello → risposta finale
- Una seconda funzione finta `leggi_email(mittente)`, che useremo come base
  del Progetto 1 (Modulo 12)
- Un mini-chatbot con più strumenti disponibili contemporaneamente

## Prerequisiti

```bash
pip install anthropic python-dotenv
```

Serve un file `.env` nella cartella del progetto con:
```
ANTHROPIC_API_KEY=sk-ant-...
```

> Questo notebook usa l'API di Anthropic (tool use) come riferimento
> principale, perché ha una sintassi molto esplicita per il flusso a due
> chiamate. Il concetto è identico con l'API di OpenAI (function calling),
> solo l'involucro sintattico cambia (vedi dispensa, sezione 2).

---
## Cella 0 — Setup: installazione e caricamento delle chiavi

In [ ]:
import os
import json
from dotenv import load_dotenv

load_dotenv()

ANTHROPIC_KEY = os.getenv("ANTHROPIC_API_KEY")

def verifica_chiave(nome: str, chiave: str | None) -> None:
    if chiave:
        print(f"OK  {nome}: {chiave[:8]}...")
    else:
        print(f"MANCA  {nome}: non trovata. Controlla il tuo file .env")

verifica_chiave("ANTHROPIC_API_KEY", ANTHROPIC_KEY)

In [ ]:
import anthropic

cliente_anthropic = anthropic.Anthropic(api_key=ANTHROPIC_KEY) if ANTHROPIC_KEY else None

# Verifica sempre l'ID modello aggiornato nella documentazione ufficiale del provider.
MODELLO = "claude-haiku-4-5-20251001"

# Prezzi indicativi in USD per milione di token (esempio illustrativo).
PREZZI = {
    "claude-haiku-4-5-20251001": {"input": 1.00, "output": 5.00},
}

def calcola_costo(tokens_input: int, tokens_output: int, modello: str) -> dict:
    """Calcola il costo in USD di una chiamata, dato il modello usato."""
    if modello not in PREZZI:
        raise ValueError(f"Modello '{modello}' non presente nella tabella prezzi.")
    prezzi = PREZZI[modello]
    costo_input = (tokens_input / 1_000_000) * prezzi["input"]
    costo_output = (tokens_output / 1_000_000) * prezzi["output"]
    return {
        "modello": modello,
        "tokens_input": tokens_input,
        "tokens_output": tokens_output,
        "costo_totale_usd": round(costo_input + costo_output, 6),
    }

print("Client pronto.")

---
## Cella 1 — La funzione finta: `cerca_url_sospetto`

Simuliamo un servizio tipo VirusTotal (fonte originale del corso) con dati
finti ma plausibili: alcuni pattern euristici molto semplici (non è un vero
motore antivirus, serve solo a dimostrare il flusso di function calling).

In [ ]:
DOMINI_NOTI_SICURI = {"anthropic.com", "wikipedia.org", "python.org", "github.com"}
TLD_SOSPETTI = {".xyz", ".top", ".click", ".zip", ".rest"}

def cerca_url_sospetto(url: str) -> dict:
    """Simula un controllo di sicurezza tipo VirusTotal su un URL.
    In un sistema reale, qui andrebbe una vera chiamata HTTP all'API esterna."""
    url_lower = url.lower()
    motivi = []

    dominio_sicuro = any(dominio in url_lower for dominio in DOMINI_NOTI_SICURI)
    tld_sospetto = any(url_lower.endswith(tld) or tld + "/" in url_lower for tld in TLD_SOSPETTI)
    ha_ip_letterale = any(url_lower.startswith(f"http://{n}.") or url_lower.startswith(f"https://{n}.") for n in "0123456789")
    e_accorciato = any(dominio in url_lower for dominio in ["bit.ly", "tinyurl", "t.co", "goo.gl"])

    if dominio_sicuro:
        verdetto = "sicuro"
    else:
        if tld_sospetto:
            motivi.append("dominio di primo livello associato spesso a campagne di spam")
        if ha_ip_letterale:
            motivi.append("l'URL punta direttamente a un indirizzo IP invece che a un dominio")
        if e_accorciato:
            motivi.append("l'URL è stato accorciato, la destinazione reale è nascosta")
        verdetto = "sospetto" if motivi else "sconosciuto"

    return {
        "url": url,
        "verdetto": verdetto,
        "motivi": motivi if motivi else ["nessun pattern sospetto rilevato (controllo euristico semplificato)"],
    }


# Prova rapida della funzione, prima di cablarla con l'LLM
print(cerca_url_sospetto("https://www.python.org/downloads/"))
print(cerca_url_sospetto("http://185.23.44.9/login-verifica.php"))
print(cerca_url_sospetto("https://bit.ly/3xY9zQa"))

---
## Cella 2 — Lo schema della funzione (JSON Schema)

Definiamo lo strumento nel formato atteso dall'API Anthropic: nome,
descrizione, e `input_schema` (JSON Schema dei parametri).

In [ ]:
STRUMENTI = [
    {
        "name": "cerca_url_sospetto",
        "description": (
            "Verifica se un URL è sospetto o pericoloso, come farebbe un "
            "servizio di analisi antivirus tipo VirusTotal. Usa questo "
            "strumento ogni volta che l'utente condivide un link e chiede "
            "se è sicuro, o quando devi valutare la sicurezza di un URL."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "url": {
                    "type": "string",
                    "description": "L'URL completo da analizzare, es. https://esempio.com/pagina",
                }
            },
            "required": ["url"],
        },
    }
]

print(json.dumps(STRUMENTI, indent=2, ensure_ascii=False))

---
## Cella 3 — Il flusso completo di function calling

Implementiamo il ciclo descritto nella dispensa (sezione 3):

1. Inviamo storico + strumenti disponibili
2. Se il modello risponde con `stop_reason == "tool_use"`, estraiamo nome e
   argomenti della funzione richiesta
3. Eseguiamo davvero la funzione nel nostro codice
4. Rimandiamo il risultato al modello come `tool_result`
5. Il modello genera la risposta finale, ora informato del risultato

**Prima di eseguire:** con quale messaggio utente ti aspetti che il modello
decida di invocare `cerca_url_sospetto`? E con quale messaggio pensi che
risponderebbe senza usare alcuno strumento?

In [ ]:
FUNZIONI_DISPONIBILI = {
    "cerca_url_sospetto": cerca_url_sospetto,
}

def chat_con_strumenti(messaggio_utente: str, storico: list[dict] | None = None) -> tuple[str, list[dict]]:
    """Gestisce un turno di conversazione con possibile chiamata a strumenti.
    Restituisce (testo_risposta_finale, storico_aggiornato)."""
    storico = storico or []
    storico.append({"role": "user", "content": messaggio_utente})

    risposta = cliente_anthropic.messages.create(
        model=MODELLO,
        max_tokens=500,
        tools=STRUMENTI,
        messages=storico,
    )

    # Il modello ha chiesto di usare uno strumento?
    while risposta.stop_reason == "tool_use":
        blocco_strumento = next(b for b in risposta.content if b.type == "tool_use")
        nome_funzione = blocco_strumento.name
        argomenti = blocco_strumento.input

        print(f"[il modello chiede di chiamare: {nome_funzione}({argomenti})]")

        if nome_funzione not in FUNZIONI_DISPONIBILI:
            risultato_funzione = {"errore": f"funzione '{nome_funzione}' non disponibile"}
        else:
            risultato_funzione = FUNZIONI_DISPONIBILI[nome_funzione](**argomenti)

        # Aggiungiamo la richiesta del modello e il risultato reale al contesto
        storico.append({"role": "assistant", "content": risposta.content})
        storico.append({
            "role": "user",
            "content": [
                {
                    "type": "tool_result",
                    "tool_use_id": blocco_strumento.id,
                    "content": json.dumps(risultato_funzione, ensure_ascii=False),
                }
            ],
        })

        risposta = cliente_anthropic.messages.create(
            model=MODELLO,
            max_tokens=500,
            tools=STRUMENTI,
            messages=storico,
        )

    testo_finale = "".join(b.text for b in risposta.content if b.type == "text")
    storico.append({"role": "assistant", "content": risposta.content})
    return testo_finale, storico


testo, storico_conversazione = chat_con_strumenti(
    "Ho ricevuto questo link via SMS: http://185.23.44.9/login-verifica.php . È sicuro cliccarci?"
)
print("\nRisposta finale:", testo)

**Le tue osservazioni** (modifica questa cella):

- Il modello ha invocato lo strumento come ti aspettavi? ...
- Prova ora con un messaggio che NON richiede lo strumento (es. "Che ore
  sono?" oppure una domanda generica) usando la cella sotto: il modello lo
  invoca comunque, o risponde direttamente?

In [ ]:
testo_senza_strumento, _ = chat_con_strumenti("Qual è la differenza tra spam e phishing, in due righe?")
print(testo_senza_strumento)

---
## Cella 4 — Seconda funzione finta: `leggi_email(mittente)`

Prepariamo il terreno per il Progetto 1 (Modulo 12): una funzione che simula
la lettura della inbox filtrata per mittente, come nel caso Gmail discusso
nella dispensa. Restituisce dati finti, ma con la stessa forma che avrebbe
una vera integrazione (dopo il filtro lato API, non lato modello).

In [ ]:
INBOX_FINTA = [
    {"mittente": "supporto@bancaesempio.it", "oggetto": "Verifica urgente del tuo conto", "corpo": "Il tuo conto è stato sospeso. Clicca qui per riattivarlo entro 24 ore: http://bancaesempio-verifica.top/login"},
    {"mittente": "newsletter@bancaesempio.it", "oggetto": "Le novità del mese", "corpo": "Scopri i nuovi tassi di interesse sui conti deposito."},
    {"mittente": "marco.rossi@aziendaesempio.it", "oggetto": "Riunione di lunedì", "corpo": "Ciao, confermi la riunione di lunedì alle 10:00?"},
    {"mittente": "supporto@bancaesempio.it", "oggetto": "Gentile cliente, aggiorna i tuoi dati", "corpo": "Per motivi di sicurezza aggiorna i tuoi dati personali qui: http://185.23.44.9/aggiorna"},
]

def leggi_email(mittente: str) -> dict:
    """Simula la lettura della inbox filtrata per mittente.
    Il filtro avviene qui (lato "API"), non lasciando che sia il modello a
    scorrere l'intera inbox: esattamente il pattern discusso nella dispensa."""
    email_trovate = [e for e in INBOX_FINTA if e["mittente"] == mittente]
    return {
        "mittente_cercato": mittente,
        "numero_email_trovate": len(email_trovate),
        "email": email_trovate,
    }


print(leggi_email("supporto@bancaesempio.it"))

---
## Cella 5 — TODO: aggiungi `leggi_email` agli strumenti disponibili

**Completa il TODO:** aggiungi la definizione dello strumento `leggi_email`
alla lista `STRUMENTI`, registra la funzione in `FUNZIONI_DISPONIBILI`, e poi
prova una richiesta che dovrebbe far scattare questo secondo strumento.

In [ ]:
# TODO 1: aggiungi la definizione dello schema per "leggi_email" a STRUMENTI.
#          Deve avere un parametro obbligatorio "mittente" di tipo string.
#          Suggerimento: copia la struttura usata per "cerca_url_sospetto" in Cella 2.

STRUMENTI_ESTESI = STRUMENTI + [
    # TODO: aggiungi qui il dizionario di definizione per "leggi_email"
]

# TODO 2: registra "leggi_email" nel dizionario delle funzioni disponibili.
FUNZIONI_DISPONIBILI_ESTESE = {
    **FUNZIONI_DISPONIBILI,
    # TODO: aggiungi qui "leggi_email": leggi_email
}


def chat_con_strumenti_estesa(messaggio_utente: str, storico: list[dict] | None = None) -> tuple[str, list[dict]]:
    """Come chat_con_strumenti, ma con l'insieme di strumenti esteso."""
    storico = storico or []
    storico.append({"role": "user", "content": messaggio_utente})

    risposta = cliente_anthropic.messages.create(
        model=MODELLO, max_tokens=500, tools=STRUMENTI_ESTESI, messages=storico
    )

    while risposta.stop_reason == "tool_use":
        blocco_strumento = next(b for b in risposta.content if b.type == "tool_use")
        nome_funzione = blocco_strumento.name
        argomenti = blocco_strumento.input
        print(f"[il modello chiede di chiamare: {nome_funzione}({argomenti})]")

        if nome_funzione not in FUNZIONI_DISPONIBILI_ESTESE:
            risultato_funzione = {"errore": f"funzione '{nome_funzione}' non disponibile"}
        else:
            risultato_funzione = FUNZIONI_DISPONIBILI_ESTESE[nome_funzione](**argomenti)

        storico.append({"role": "assistant", "content": risposta.content})
        storico.append({
            "role": "user",
            "content": [{"type": "tool_result", "tool_use_id": blocco_strumento.id,
                         "content": json.dumps(risultato_funzione, ensure_ascii=False)}],
        })
        risposta = cliente_anthropic.messages.create(
            model=MODELLO, max_tokens=500, tools=STRUMENTI_ESTESI, messages=storico
        )

    testo_finale = "".join(b.text for b in risposta.content if b.type == "text")
    storico.append({"role": "assistant", "content": risposta.content})
    return testo_finale, storico


# Dopo aver completato i TODO sopra, prova:
# testo, _ = chat_con_strumenti_estesa("Leggi le email ricevute da supporto@bancaesempio.it e dimmi se ce n'è qualcuna sospetta.")
# print(testo)

---
## Reto finale — Assistente di sicurezza con più strumenti

Costruisci un piccolo assistente conversazionale che, sullo stesso storico,
può decidere in autonomia se e quando usare `cerca_url_sospetto`,
`leggi_email`, o nessuno dei due — a seconda di cosa chiede l'utente in ogni
turno. Usa `chat_con_strumenti_estesa` come base (dopo aver completato i TODO
della cella precedente) e traccia il costo totale della sessione con
`calcola_costo`.

In [ ]:
def sessione_sicurezza(messaggi_utente: list[str]) -> None:
    """Esegue una sequenza di messaggi utente sullo stesso storico,
    lasciando che il modello scelga quando usare gli strumenti disponibili,
    e stampa un riepilogo del costo totale."""
    storico: list[dict] = []
    costo_totale_usd = 0.0

    for messaggio in messaggi_utente:
        print(f"\nUtente: {messaggio}")
        testo, storico = chat_con_strumenti_estesa(messaggio, storico)
        print(f"Assistente: {testo}")

        # Nota: per stimare il costo con precisione turno per turno servirebbe
        # tracciare risposta.usage ad ogni chiamata interna. Come esercizio
        # aggiuntivo, modifica chat_con_strumenti_estesa perché restituisca
        # anche i token usati, e accumula qui il costo con calcola_costo().

    print(f"\n[sessione conclusa — {len(messaggi_utente)} turni]")


sessione_sicurezza([
    "Ciao, puoi controllare se questo link è affidabile? https://bit.ly/3xY9zQa",
    "Ora leggi le email di marco.rossi@aziendaesempio.it e dimmi di cosa parlano.",
])

---
## Autovalutazione della settimana

1. **Cosa ha funzionato bene questa settimana?**
   ...

2. **Cosa non hai capito del tutto?**
   ...

3. **Cosa cambieresti nel tuo approccio la prossima settimana?**
   ...

4. **Tempo reale dedicato:**
   ...

---
*Notebook 11 — Modulo 11 del Corso Completo di IA Generativa*